# Summarize Evaluation Results

In [1]:
import json
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("../outputs/evaluation")

RUNS = {
    "Base model + persona": (
        OUTPUT_DIR
        / "qwen25_05b_base_5000_eval"
        / "summary.json"
    ),
    "Fine-tuned model + persona": (
        OUTPUT_DIR
        / "qwen25_05b_compact_5000_eval"
        / "summary.json"
    ),
    "Base model, no persona": (
        OUTPUT_DIR
        / "qwen25_05b_no_persona_base_5000"
        / "summary.json"
    ),
    "Fine-tuned model, no persona": (
        OUTPUT_DIR
        / "qwen25_05b_no_persona_5000"
        / "summary.json"
    ),
}

In [2]:
def load_evaluation_summary(
    model_name: str,
    summary_path: Path,
) -> dict:
    """Load the main metrics from one evaluation summary."""

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Evaluation summary not found: {summary_path}"
        )

    with summary_path.open(
        encoding="utf-8"
    ) as file:
        summary = json.load(file)

    return {
        "model": model_name,
        "model_type": summary["model_type"],
        "examples": summary["examples"],
        "valid_json_pct": (
            100 * summary["valid_json_rate"]
        ),
        "format_repair_pct": (
            100 * summary["format_repair_rate"]
        ),
        "valid_schema_pct": (
            100 * summary["valid_schema_rate"]
        ),
        "scoreable_response_pct": (
            100 * summary["scoreable_response_rate"]
        ),
        "exact_match_pct": (
            100 * summary["exact_match_accuracy"]
        ),
        "normalized_accuracy_pct": (
            100 * summary["normalized_accuracy"]
        ),
        "task_weighted_accuracy_pct": (
            100
            * summary[
                "task_weighted_normalized_accuracy"
            ]
        ),
        "task_weighted_accuracy_including_invalid_pct": (
            100
            * summary[
                "task_weighted_normalized_accuracy_including_invalid"
            ]
        ),
        "scored_responses": summary[
            "scored_responses"
        ],
        "eligible_responses": summary[
            "eligible_responses"
        ],
        "scored_tasks": summary["scored_tasks"],
    }

In [3]:
model_results = pd.DataFrame(
    [
        load_evaluation_summary(
            model_name,
            summary_path,
        )
        for model_name, summary_path in RUNS.items()
    ]
).set_index("model")

model_results.round(2)

,model_type,examples,valid_json_pct,format_repair_pct,valid_schema_pct,scoreable_response_pct,exact_match_pct,normalized_accuracy_pct,task_weighted_accuracy_pct,task_weighted_accuracy_including_invalid_pct,scored_responses,eligible_responses,scored_tasks
model,,,,,,,,,,,,,
Base model + persona,base,5000,0.04,98.96,97.10,82.62,40.40,31.74,49.73,39.47,6093,7375,14
Fine-tuned model + persona,lora,5000,100.00,0.00,100.00,100.00,55.52,73.71,71.75,71.75,7375,7375,17
"Base model, no persona",base,5000,5.98,93.28,96.32,62.63,18.84,7.48,28.60,21.78,4619,7375,13
"Fine-tuned model, no persona",lora,5000,100.00,0.00,100.00,100.00,55.06,73.64,71.59,71.59,7375,7375,17


## Overall prediction performance

In [4]:

overall_performance = model_results[
    [
        "exact_match_pct",
        "task_weighted_accuracy_pct",
        "task_weighted_accuracy_including_invalid_pct",
    ]
].rename(
    columns={
        "exact_match_pct": "Exact match (%)",
        "task_weighted_accuracy_pct": (
            "Task-weighted normalized accuracy (%)"
        ),
        "task_weighted_accuracy_including_invalid_pct": (
            "Primary accuracy including invalid (%)"
        ),
    }
)


display(
    overall_performance.style.format(
        {
            "Exact match (%)": "{:.2f}",
            "Task-weighted normalized accuracy (%)": (
                "{:.2f}"
            ),
            "Primary accuracy including invalid (%)": (
                "{:.2f}"
            ),
        },
        na_rep="—",
    )
)

,Exact match (%),Task-weighted normalized accuracy (%),Primary accuracy including invalid (%)
model,,,
Base model + persona,40.40,49.73,39.47
Fine-tuned model + persona,55.52,71.75,71.75
"Base model, no persona",18.84,28.60,21.78
"Fine-tuned model, no persona",55.06,71.59,71.59


## Formatting accuracy

In [5]:
formatting_accuracy = model_results[
    [
        "valid_json_pct",
        "format_repair_pct",
        "valid_schema_pct",
    ]
].rename(
    columns={
        "valid_json_pct": "Valid JSON (%)",
        "format_repair_pct": "Format repaired (%)",
        "valid_schema_pct": "Valid schema (%)",
    }
)

display(
    formatting_accuracy.style.format(
        "{:.2f}"
    )
)

,Valid JSON (%),Format repaired (%),Valid schema (%)
model,,,
Base model + persona,0.04,98.96,97.10
Fine-tuned model + persona,100.00,0.00,100.00
"Base model, no persona",5.98,93.28,96.32
"Fine-tuned model, no persona",100.00,0.00,100.00


## Coverage

In [6]:
TOTAL_TASKS = 17

coverage = model_results[
    [
        "examples",
        "scored_responses",
        "eligible_responses",
        "scoreable_response_pct",
        "scored_tasks",
    ]
].rename(
    columns={
        "examples": "Question examples",
        "scored_responses": "Scored responses",
        "eligible_responses": "Eligible responses",
        "scoreable_response_pct": "Scoreable rate (%)",
        "scored_tasks": "Scored tasks",
    }
)

coverage["Scored tasks"] = coverage[
    "Scored tasks"
].map(
    lambda value: f"{int(value)} / {TOTAL_TASKS}"
)

display(
    coverage.style.format(
        {
            "Question examples": "{:,.0f}",
            "Scored responses": "{:,.0f}",
            "Eligible responses": "{:,.0f}",
            "Scoreable rate (%)": "{:.2f}",
        }
    )
)

,Question examples,Scored responses,Eligible responses,Scoreable rate (%),Scored tasks
model,,,,,
Base model + persona,"5,000","6,093","7,375",82.62,14 / 17
Fine-tuned model + persona,"5,000","7,375","7,375",100.00,17 / 17
"Base model, no persona","5,000","4,619","7,375",62.63,13 / 17
"Fine-tuned model, no persona","5,000","7,375","7,375",100.00,17 / 17


## RQ1: Effect of persona information

In [7]:
PRIMARY_METRIC = (
    "task_weighted_accuracy_including_invalid_pct"
)

persona_effect = pd.DataFrame(
    [
        {
            "Model": "Base model",
            "Without persona (%)": model_results.loc[
                "Base model, no persona",
                PRIMARY_METRIC,
            ],
            "With persona (%)": model_results.loc[
                "Base model + persona",
                PRIMARY_METRIC,
            ],
        },
        {
            "Model": "Fine-tuned model",
            "Without persona (%)": model_results.loc[
                "Fine-tuned model, no persona",
                PRIMARY_METRIC,
            ],
            "With persona (%)": model_results.loc[
                "Fine-tuned model + persona",
                PRIMARY_METRIC,
            ],
        },
    ]
).set_index("Model")

persona_effect["Persona effect (%)"] = (
    persona_effect["With persona (%)"]
    - persona_effect["Without persona (%)"]
)

display(
    persona_effect.style.format(
        "{:.2f}"
    )
)

,Without persona (%),With persona (%),Persona effect (%)
Model,,,
Base model,21.78,39.47,17.69
Fine-tuned model,71.59,71.75,0.16


## RQ2: Effect of supervised fine-tuning

In [8]:
finetuning_effect = pd.DataFrame(
    [
        {
            "Input condition": "Without persona",
            "Base model (%)": model_results.loc[
                "Base model, no persona",
                PRIMARY_METRIC,
            ],
            "Fine-tuned model (%)": model_results.loc[
                "Fine-tuned model, no persona",
                PRIMARY_METRIC,
            ],
        },
        {
            "Input condition": "With persona",
            "Base model (%)": model_results.loc[
                "Base model + persona",
                PRIMARY_METRIC,
            ],
            "Fine-tuned model (%)": model_results.loc[
                "Fine-tuned model + persona",
                PRIMARY_METRIC,
            ],
        },
    ]
).set_index("Input condition")

finetuning_effect["Fine-tuning effect (%)"] = (
    finetuning_effect["Fine-tuned model (%)"]
    - finetuning_effect["Base model (%)"]
)

display(
    finetuning_effect.style.format(
        "{:.2f}"
    )
)

,Base model (%),Fine-tuned model (%),Fine-tuning effect (%)
Input condition,,,
Without persona,21.78,71.59,49.82
With persona,39.47,71.75,32.28
